In [5]:
import torch
import torchvision.datasets as dsets
import torchvision.transforms as transforms
import torch.nn.init

In [6]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(12)
if device == 'cuda':
    torch.cuda.manual_seed_all(12)

In [7]:
LEARNING_RATE = 0.001
TRAIN_EPOCHS = 15
BATCH_SIZE = 100

In [8]:
mnist_train = dsets.MNIST(root='../dataset/', train=True, transform=transforms.ToTensor(), download=True)
mnist_test = dsets.MNIST(root='../dataset/', train=False, transform=transforms.ToTensor(), download=True)
print(mnist_train.__getitem__(1))

(tensor([[[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000

In [9]:
data_loader = torch.utils.data.DataLoader(dataset=mnist_train,
                                          batch_size=BATCH_SIZE,
                                          shuffle=True,
                                          drop_last=True)

In [10]:
class CNN(torch.nn.Module):

    def __init__(self):
        super(CNN, self).__init__()
        
        # 입력: (batch, 1, 28, 28)
        # Conv: (batch, 32, 28, 28)  => padding=1이 있어 크기가 유지됨, 1채널->32채널
        # ReLU: 비선형 활성화 함수
        # MaxPool: (batch, 32, 14, 14) => 크기가 절반이됨
        self.layer1 = torch.nn.Sequential(
            torch.nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size=2, stride=2))

       
        self.layer2 = torch.nn.Sequential(
            torch.nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size=2, stride=2))

        # 7x7크기에 64개의 채널을 flatten 시킴, 10가지 숫자로 분류
        self.fc = torch.nn.Linear(7 * 7 * 64, 10, bias=True)

        # 분류층의 가중치를 Xavier 초기화
        torch.nn.init.xavier_uniform_(self.fc.weight)

    # Conv → ReLU → Pool → Conv → ReLU → Pool
    # 출력은 (batch, 10). 아직 소프트맥스가 적용되어 있지 않음
    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = out.view(out.size(0), -1) 
        out = self.fc(out)
        return out


In [11]:
model = CNN().to(device)
criterion = torch.nn.CrossEntropyLoss().to(device)    # nn.CrossEntropyLoss()는 Softmax가 포함되어 있음
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [ ]:
total_batch = len(data_loader) # batch size=100 이고 total_batch=600이므로 총 데이터는 60000개

# 모델에 학습되는 데이터값은 0~1범위임
for epoch in range(TRAIN_EPOCHS):
    avg_cost = 0  # 에포크당 평균 비용. 한 에포크 = 전체데이터셋을 전부 사용 (60000개)

    for X, Y in data_loader:  # 미니 배치 단위로 데이터를 꺼내옴. X는 입력 데이터, Y는 레이블
        X = X.to(device)
        Y = Y.to(device) 

        optimizer.zero_grad()  
        hypothesis = model(X)
        cost = criterion(hypothesis, Y)  # 배치사이즈 100개에 대한 평균임
        cost.backward() 
        optimizer.step()

        avg_cost += cost / total_batch  #배치의 수 600으로 다시 나눔

    
    print(f'Epoch: {epoch+1}, cost = {avg_cost:.7}')


Epoch: 1, cost = 0.2331606
Epoch: 2, cost = 0.06389987
Epoch: 3, cost = 0.04566337
Epoch: 4, cost = 0.03604722
Epoch: 5, cost = 0.02975929
Epoch: 6, cost = 0.02559823
Epoch: 7, cost = 0.02177136
Epoch: 8, cost = 0.01811454
Epoch: 9, cost = 0.01524242
Epoch: 10, cost = 0.01250638
Epoch: 11, cost = 0.0110374
Epoch: 12, cost = 0.009925807
Epoch: 13, cost = 0.008925666
Epoch: 14, cost = 0.00636563
Epoch: 15, cost = 0.00650561


In [15]:
with torch.no_grad():
    
    X_test = mnist_test.test_data.view(len(mnist_test), 1, 28, 28).float().to(device)
    Y_test = mnist_test.test_labels.to(device)
    prediction = model(X_test)  # 테스트 데이터에 대해 모델이 예측한 결과값
    correct = torch.argmax(prediction, 1) == Y_test  # 가장 값이 큰 인덱스가 Y_test와 일치할 경우 맞는 예측
    accuracy = correct.float().mean()
    print(accuracy.item())
    

0.988099992275238


In [13]:
scripted_model = torch.jit.script(model)
scripted_model.save("../models/mnist_scripted_model.pt")